In [ ]:
import pandas as pd
import numpy as np
import re

form_df = pd.read_excel('Ecowatt form (Responses).xlsx', sheet_name='Form Responses 1')

def extract_english(text):
    if pd.isna(text): return text
    return str(text).split('|')[0].strip()

month_map = {
    'January': 1, 'يناير': 1,
    'February': 2, 'فبراير': 2,
    'March': 3, 'مارس': 3,
    'April': 4, 'أبريل': 4, 'ابريل': 4,
    'May': 5, 'مايو': 5,
    'June': 6, 'يونيو': 6,
    'July': 7, 'يوليو': 7,
    'August': 8, 'أغسطس': 8, 'اغسطس': 8,
    'September': 9, 'سبتمبر': 9,
    'October': 10, 'أكتوبر': 10, 'اكتوبر': 10,
    'November': 11, 'نوفمبر': 11,
    'December': 12, 'ديسمبر': 12
}

def clean_month(text):
    if pd.isna(text): return np.nan
    text_str = str(text).strip()
    for k, v in month_map.items():
        if k.lower() in text_str.lower():
            return v
    return np.nan

def clean_arabic_numerals(text):
    if pd.isna(text): return text
    text = str(text)
    arabic_to_english = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
    text = text.translate(arabic_to_english)
    text = text.replace(',', '.')
    text = re.sub(r'[^\d.-]', '', text)
    try:
        return float(text)
    except:
        return np.nan

def clean_year(year_val):
    val = clean_arabic_numerals(year_val)
    if pd.isna(val): return 2026
    if val >= 1445 and val <= 1450: return 2026
    if val < 2000: return 2026
    return int(val)

records = []
user_id_start = 300
household_id_start = 3000
bill_id_start = 20000

for idx, row in form_df.iterrows():
    u_id = user_id_start + idx
    h_id = household_id_start + idx * 10

    region = extract_english(row.iloc[1])
    housing = extract_english(row.iloc[2])
    built_up_area = clean_arabic_numerals(row.iloc[3])
    residents = clean_arabic_numerals(row.iloc[4])
    cooking = extract_english(row.iloc[5])
    ac = extract_english(row.iloc[6])
    ac_set_temp = extract_english(row.iloc[7])
    ac_off_when_leaving = extract_english(row.iloc[8])
    insulation = True if extract_english(row.iloc[9]) == 'Yes' else False
    applies_energy_tips = extract_english(row.iloc[10])
    willing_to_upgrade = extract_english(row.iloc[11])
    energy_saving_interest = extract_english(row.iloc[12])
    energy_price_pressure = extract_english(row.iloc[13])
    home_improvement_interest = extract_english(row.iloc[14])
    encouraged_by_family = extract_english(row.iloc[15])
    indoor_thermal_adjustment = extract_english(row.iloc[16])
    ac_off_for_one_hour = extract_english(row.iloc[17])
    home_occupancy_time = extract_english(row.iloc[18])
    ac_operating_times = extract_english(row.iloc[19])

    household_records = []
    user_month_year_seen = set()

    for b in range(6):
        month_idx = 20 + b * 4
        year_idx = 21 + b * 4
        kwh_idx = 22 + b * 4
        cost_idx = 23 + b * 4

        m_raw = row.iloc[month_idx]
        y_raw = row.iloc[year_idx]
        kwh_raw = row.iloc[kwh_idx]
        cost_raw = row.iloc[cost_idx]

        m = clean_month(m_raw)
        if pd.isna(m): continue

        y = clean_year(y_raw)
        kwh = clean_arabic_numerals(kwh_raw)
        cost = clean_arabic_numerals(cost_raw)

        if pd.isna(cost): continue
        if pd.isna(kwh):
            kwh = cost / 0.18

        dup_key = (u_id, y, m)
        if dup_key in user_month_year_seen:
            continue
        user_month_year_seen.add(dup_key)

        season = 'Winter' if m in [12, 1, 2] else 'Rest_of_Year'

        household_records.append({
            'user_id': u_id,
            'household_id': h_id,
            'bill_id': bill_id_start + len(household_records),
            'region': region,
            'housingType': housing,
            'builtUpArea': built_up_area,
            'numberOfResidents': residents,
            'cookingFuel': cooking,
            'acType': ac,
            'hasInsulation': insulation,
            'acSetTemp': ac_set_temp,
            'acOffWhenLeaving': ac_off_when_leaving,
            'appliesEnergyTips': applies_energy_tips,
            'willingToUpgradeAppliance': willing_to_upgrade,
            'energySavingInterest': energy_saving_interest,
            'energyPricePressure': energy_price_pressure,
            'homeImprovementInterest': home_improvement_interest,
            'encouragedByFamily': encouraged_by_family,
            'indoorThermalAdjustment': indoor_thermal_adjustment,
            'acOffForOneHour': ac_off_for_one_hour,
            'homeOccupancyTime': home_occupancy_time,
            'acOperatingTimes': ac_operating_times,
            'month': int(m),
            'year': int(y),
            'season': season,
            'totalKwh': round(kwh, 2),
            'totalCostSar': round(cost, 2)
        })

    if len(household_records) >= 2:
        records.extend(household_records)

out_df = pd.DataFrame(records)
output_file = 'user_profile.csv'
out_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print("Done! File saved as user_profile.csv with shape:", out_df.shape)

Done! File saved as user_profile.csv with shape: (23, 27)
